# SelfSNE quick demo

Minimal Self-Supervised Noise Embedding training on synthetic data.

In [ ]:
import torch
import pytorch_lightning as pl
from torch.utils.data import DataLoader, TensorDataset
from selfsne import SelfSNE
from selfsne.losses import LikelihoodRatioEstimator

In [ ]:
pl.seed_everything(0)
t = torch.linspace(0, 4*torch.pi, 256)
x = torch.stack([t.sin(), t.cos(), (2*t).sin(), (2*t).cos()], dim=1)
ds = TensorDataset(x[:-1], x[1:])
loader = DataLoader(ds, batch_size=64)

In [ ]:
model = SelfSNE(
    loss=LikelihoodRatioEstimator(divergence="bce"),
    target_encoder=torch.nn.Sequential(
        torch.nn.Linear(4, 32), torch.nn.ReLU(), torch.nn.Linear(32, 2)
    )
)

trainer = pl.Trainer(max_epochs=3, accelerator="cpu", devices=1, logger=False, enable_checkpointing=False, enable_progress_bar=False)
trainer.fit(model, loader)
emb = trainer.predict(model, loader)
embeddings = torch.cat([b["target_embedding"] for b in emb])
print("Embeddings shape:", embeddings.shape)

In [ ]:
# SelfSNE MI lower bound via likelihood ratio estimation
# The LikelihoodRatioEstimator provides a variational lower bound on I(X;Y)
# We approximate it by the mean log-ratio on held-out pairs
import torch
import numpy as np
mi_estimates = []
for xb, yb in loader:
    with torch.no_grad():
        ctx_emb, tgt_emb, _, _ = model.get_embeddings(xb, yb, None)
        logits = model.loss.logits(ctx_emb, tgt_emb)
        mi_estimates.append(logits['pos_logits'].mean().item())
mi_lb = float(np.mean(mi_estimates))
print("SelfSNE MI lower bound estimate:", mi_lb)

## MI estimation test with known Gaussian MI

Generate bivariate Gaussian with known correlation rho, compare true MI to SelfSNE lower bound.

In [ ]:
import numpy as np
from torch.utils.data import DataLoader
rho = 0.7
n = 20000
mean = [0,0]
cov = [[1, rho],[rho,1]]
xy = np.random.multivariate_normal(mean, cov, size=n)
x = torch.from_numpy(xy[:,:1].astype(np.float32))
y = torch.from_numpy(xy[:,1:].astype(np.float32))
ds = TensorDataset(x, y)
loader_gauss = DataLoader(ds, batch_size=512, shuffle=True)

In [ ]:
true_mi = -0.5 * np.log(1 - rho**2)
print("True Gaussian MI:", true_mi)

In [ ]:
from selfsne.nn.norm import InputNorm1d
pl.seed_everything(1)
encoder = torch.nn.Sequential(
    InputNorm1d(1),
    torch.nn.Linear(1, 128),
    torch.nn.SELU(),
    torch.nn.Linear(128, 64),
    torch.nn.SELU(),
    torch.nn.Linear(64, 2)
)
model_g = SelfSNE(
    loss=LikelihoodRatioEstimator(divergence="bce", baseline="parametric"),
    target_encoder=encoder,
    learning_rate=0.003,
    lr_scheduler=True,
    lr_warmup_steps=500,
    lr_target_steps=2000,
    lr_cosine_steps=2000
)
trainer_g = pl.Trainer(max_epochs=50, accelerator="cpu", devices=1, logger=False, enable_checkpointing=False, enable_progress_bar=False)
trainer_g.fit(model_g, loader_gauss)

In [ ]:
mi_estimates_g = []
for xb, yb in loader_gauss:
    with torch.no_grad():
        ctx_emb, tgt_emb, _, _ = model_g.get_embeddings(xb, yb, None)
        logits = model_g.loss.logits(ctx_emb, tgt_emb)
        mi_estimates_g.append(logits['pos_logits'].mean().item())
mi_lb_g = float(np.mean(mi_estimates_g))
print("SelfSNE MI lower bound estimate:", mi_lb_g)
print("True MI:", true_mi)

In [ ]:
import matplotlib.pyplot as plt
plt.figure()
plt.bar(["True MI", "SelfSNE LB"], [true_mi, mi_lb_g])
plt.ylabel("MI")
plt.title("MI estimation: Gaussian ground truth vs SelfSNE lower bound")
plt.show()